Initialization


In [5]:
import numpy as np
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1' # Suppress info messages from TensorFlow
import tensorflow as tf

import h5py

import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats('svg')
from matplotlib.lines import Line2D
import datetime

from keras.models import Sequential
from keras.layers import Dense, Activation
tf.keras.backend.set_floatx('float64')

from scipy.signal import savgol_filter


basic functions

In [6]:
def getCurrentdate_time():
    d = datetime.datetime.now()
    return d.strftime("%a %d/%m/%y %H:%M:%S")

print(getCurrentdate_time())

Tue 29/09/26 19:41:07


Functions related to H5file

In [7]:
def WriteH5(filename , name , depsp1,T1,datas): #name refers to activation???  
    h5f = h5py.File(filename, 'w')
    h5f.attrs['Date'] = getCurrentdate_time()
    h5f.attrs['Name'] = name
    h5f.create_dataset('depsp', data = depsp1, dtype = 'float64')
    h5f.create_dataset('T', data = T1, dtype= 'float64')
    h5fData = h5f.create_group('Data')
    h5fData.attrs['Number'] = len(datas)
    for data_row in datas:
        name = str(data_row[0]) +'_' + str(data_row[1])
        curData = h5fData.create_dataset(name, data = data_row[2],dtype = 'float64' )
        curData.attrs['depsp'] = data_row[0]
        curData.attrs['T'] = data_row[1]
    h5f.close()


#WriteH5('first_testh5.h5', 'idontknow',0.1,1000,datas)


def readH5(filename):
    h5f = h5py.File(filename, 'r')
    print('Date_of_the_file: ',h5f.attrs['Date'])
    print('Name_of_the_file:', h5f.attrs['Name'])
    dataH5 = h5f['Data']
    print('contains', dataH5.attrs['Number'],'group of data')
    d1 = list(dataH5.keys())  # all the dataset and group in the first layer, prior group
    if (len(d1) != dataH5.attrs['Number']):
        print("size error!!!")
    else:
        print("size seems ok")
    datas = []
    for data_group_name in d1:
        data = dataH5[data_group_name]
        depsp = data.attrs['depsp']
        T = data.attrs['T']
        name = str(depsp) + '_' + str(T)
        datas.append([depsp,T,data[:]])
    h5f.close()
    return datas
    

    

function to create ANNH5 File

In [8]:
def writeArrayH5(file, name, data):
    with h5py.File(file, 'a') as hf:
        try:
            del hf[name]
        except:
            pass
        hf.create_dataset(name,  data=data)

def readArrayH5(file, name):
    with h5py.File(file, 'r') as hf:
        data = hf[name][:]
    return data


def WriteANNH5(filename):
    with h5py.File(filename, 'w') as hf:
        i = 1
        hf.create_dataset('layers', data = np.array([len(model.layers)]))
        for layer in model.layers:
            hf.create_dataset('w'+str(i), data= layer.get_weights()[0])
            hf.create_dataset('b'+str(i), data = layer.get_weights()[1])
            i = i+1
        hf.create_dataset('minData', data = minData )
        hf.create_dataset('maxData', data = maxData )
        hf.create_dataset('deps0', data = np.array([deps0]))

'''minData = np.array([0.0, 0.0, 0.0])
maxData = np.array([1.0, 1.0, 1.0])
deps0   = 1e-3
model = tf.keras.Sequential([
    tf.keras.layers.Dense(15, activation='exponential', input_shape=(3,)),
    tf.keras.layers.Dense(7,  activation='exponential'),
    tf.keras.layers.Dense(1,  activation='linear'),
])
'''

#WriteANNH5('ANNtesth5.h5')

def ReadANNH5(filename):   
    with h5py.File(filename, 'r') as hf:
        layers = hf['layers'][:][0]
        #print(layers)
        w = []
        b = []
        r = []
        for i in range(layers):
            w.append(hf['w'+str(i+1)][:].T)
            b.append(hf['b'+str(i+1)][:][:,np.newaxis])       
        r.append(hf['minData'][:][:,np.newaxis])
        r.append(hf['maxData'][:][:,np.newaxis])
        r.append(hf['deps0'][:][0])
        return w,b,r

#print(ReadANNH5('ANNtesth5.h5')[2])




Data transformation

In [9]:
# depsp
def getDepspArray(datas):
    l = []
    for row in datas:
        l.append(row[0])
    return np.unique(np.array(l))

# T
def getTArray(datas):
    l = []
    for row in datas:
        l.append(row[1]) 
    return np.unique(np.array(l))


    
# write_cvs
def writeTXT(name,d):
    n = name + '_' + str(d[0]) + '_' + str(d[1])
    np.savetxt(n, d[2], delimiter=' ', header=str(d[0]) + '\n' + str(d[1]))

# read cvs
def readTXT(name):
    d = np.loadtxt(name)
    f = open(name, 'r')
    l1 = f.readline()
    l2 = f.readline()
    f.close()
    e = float(l1.rsplit()[1])
    T = float(l2.rsplit()[1])
    return [e, T, d]

# write cvss
def writeTXTs(name, datas):
    for d in datas:
        writeTXT(name, d)

# read cvss
def readTXTs(name):
    table = np.loadtxt(name)
    datas = []
    for row in table:
        d = [row[0],row[1],row[2:]]
        datas.append(d)
    return datas





plot the code


In [ ]:
colors = ['#7CB5EC','#F7A35C','#A2A2D0','#F6675D','#47ADC7']


def create_dummy_line(**kwds):
    return Line2D([], [], **kwds)
    
baseSize = (8, 6)  # Base size of a subplot

def sbPlot(n):
    if (n == 1): return 1, 1
    if (n <= 2): return 1, 2
    if (n <= 4): return 2, 2
    if (n <= 6): return 3, 2
    if (n <= 9): return 3, 3
    if (n <= 12): return 4, 3
    return 0, 0

def sbPlotSize(n):
    x, y = sbPlot(n)
    return baseSize[0] * y, baseSize[1] * x
    
def plotDatas(datas, filename=''):
    Ts = getTArray(datas)
    depsps = getDepspArray(datas)
    ndepsp = depsps.shape[0]
    nT = Ts.shape[0]
    
    plt.figure(figsize = sbPlotSize(ndepsp))
    plt.rc('text', usetex = True)
    plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
    plt.subplots_adjust(hspace = 0.3)
    xs, ys = sbPlot(ndepsp)
    idx = 1
    for depsp in depsps:
        plt.subplot(xs, ys, idx)
        cl = 0
        for T in Ts:
            for d in datas:
                if (d[0] == depsp and d[1] == T):
                    subdata = d[2]
                    arr = np.linspace(0, 0.02 * int(len(subdata)-1), num = len(subdata))
                    plt.plot(arr, subdata, label=r'$T=' + str(d[1]) + '^{\circ}C$', linewidth = 2, color=colors[cl])
            cl += 1
        plt.xlabel(r'strain $\varepsilon$', fontsize = 16) # Labels the x axis
        plt.ylabel(r'flow stress $\sigma$ (MPa)', fontsize = 16) # Labels the y axis
        plt.title(r'strain rate $\dot{\varepsilon} = ' + str(depsp) + '$ s$^{-1}$', fontsize = 16) # Self explicit command
        plt.xlim(arr.min(), arr.max())
        #plt.ylim(bottom = 0)
        idx += 1
    legendLines = []
    cl = 0
    for temp in list(Ts):
        legendLines.append((r'$T=$' + str(int(temp)) + r'$^{\circ}$C', {'color':colors[cl], 'linestyle':'-', 'linewidth':2.5, 'marker':'s'}))
        cl += 1
    
    if (ndepsp % 2):
        plt.legend([create_dummy_line(**l[1]) for l in legendLines], [l[0] for l in legendLines], 
               loc = 'upper center', fontsize = 12, ncols = 6, bbox_to_anchor = (1.0, -0.2), shadow = False)
    else:
        plt.legend([create_dummy_line(**l[1]) for l in legendLines], [l[0] for l in legendLines], 
               loc = 'upper center', fontsize = 12, ncols = 6, bbox_to_anchor = (0.0, -0.2), shadow = False)
            
    if (filename!=''):
        plt.savefig(filename, bbox_inches='tight')
    plt.show()


[0.   0.02 0.04 0.06 0.08 0.1  0.12 0.14 0.16 0.18 0.2  0.22 0.24 0.26
 0.28 0.3  0.32 0.34 0.36 0.38 0.4  0.42 0.44 0.46 0.48 0.5  0.52 0.54
 0.56 0.58 0.6 ]
